In [1]:
import pandas as pd
import re

from pyparsing import col

In [2]:
raw_df = pd.read_csv('stops.csv')

In [3]:
raw_df[raw_df["stop_name"].str.contains(r'^(?:Downsview Park Station)', case=False, na=False)]

raw_df.shape



(9393, 12)

In [4]:
df = raw_df.copy()

In [5]:
df.head

<bound method NDFrame.head of       stop_id  stop_code                                     stop_name  \
0         662        662                     Danforth Rd at Kennedy Rd   
1         929        929                    Davenport Rd at Bedford Rd   
2         940        940                     Davenport Rd at Dupont St   
3        1871       1871                Davisville Ave at Cleveland St   
4       11700      11700                        Disco Rd at Attwell Dr   
...       ...        ...                                           ...   
9388    16859      16859              CARLTON ST AT YONGE ST EAST SIDE   
9389    16860      16860           QUEEN ST EAST AT RIVER ST WEST SIDE   
9390    16861      16861          SPADINA AVE AT COLLEGE ST NORTH SIDE   
9391    16862      16862           COLLEGE ST AT SPADINA AVE EAST SIDE   
9392    16863      16863  ST MATTHEWS RD AT GERRARD ST EAST NORTH SIDE   

      stop_desc   stop_lat   stop_lon  zone_id  stop_url  location_type  \
0     

In [6]:
df.shape

(9393, 12)

In [7]:
# df = df[
#     (
#             df['stop_name'].str.contains(r'\b(?:terminal|station)\b', case=False, na=False) |
#             df['stop_name'].str.contains(r'\bYork University\b', case=False, na=False)
#     )
#     &
#     ~df['stop_name'].str.contains(r'\b(?:at|go)\b', case=False, na=False)
#     ]

In [8]:
df[df["stop_name"].str.contains(r'\b(?:North York Centre Station)\b', case=False, na=False)]

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
6321,8984,8984,Yonge St at Empress Ave - North York Centre St...,NaN,43.769149,-79.412700,NaN,NaN,NaN,NaN,NaN,1
8058,13789,13789,North York Centre Station - Southbound Platform,NaN,43.767347,-79.412492,NaN,NaN,NaN,NaN,NaN,1
8119,13790,13790,North York Centre Station - Northbound Platform,NaN,43.768547,-79.412592,NaN,NaN,NaN,NaN,NaN,1


In [9]:
df[df["stop_name"].str.contains(r'\b(?:York University)\b', case=False, na=False)]

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
1704,11330,11330,Bayview Ave at Lawrence Ave East - York Univer...,NaN,43.727687,-79.382237,NaN,NaN,NaN,NaN,NaN,1
6143,1595,1595,Steeles Ave West at Founders Rd - York University,NaN,43.779428,-79.504234,NaN,NaN,NaN,NaN,NaN,1
6152,1726,1726,Bayview Ave at Lawrence Ave East - York Univer...,NaN,43.727267,-79.381568,NaN,NaN,NaN,NaN,NaN,1
6326,1594,1594,Steeles Ave West at Founders Rd East Side - Yo...,NaN,43.779212,-79.503758,NaN,NaN,NaN,NaN,NaN,1
8521,15666,15666,York University - Northbound Platform,NaN,43.774097,-79.499888,NaN,NaN,NaN,NaN,NaN,1
8522,15667,15667,York University - Southbound Platform,NaN,43.774097,-79.499888,NaN,NaN,NaN,NaN,NaN,1


In [10]:
df.shape

(9393, 12)

In [11]:
len(df['stop_name'].unique())

7558

In [12]:
df.stop_name

0                          Danforth Rd at Kennedy Rd
1                         Davenport Rd at Bedford Rd
2                          Davenport Rd at Dupont St
3                     Davisville Ave at Cleveland St
4                             Disco Rd at Attwell Dr
                            ...                     
9388                CARLTON ST AT YONGE ST EAST SIDE
9389             QUEEN ST EAST AT RIVER ST WEST SIDE
9390            SPADINA AVE AT COLLEGE ST NORTH SIDE
9391             COLLEGE ST AT SPADINA AVE EAST SIDE
9392    ST MATTHEWS RD AT GERRARD ST EAST NORTH SIDE
Name: stop_name, Length: 9393, dtype: str

In [13]:
len(df.stop_name.unique())

7558

In [14]:

ttc_stations = {
    1: ["Finch Station", "North York Centre Station", "Sheppard-Yonge Station", "York Mills Station",
        "Lawrence Station", "Eglinton Station", "Davisville Station", "St Clair Station", "Summerhill Station",
        "Rosedale Station", "Bloor Station", "Wellesley Station", "College Station", "TMU Station", "Queen Station",
        "King Station", "Union Station", "St Andrew Station", "Osgoode Station", "St Patrick Station",
        "Queen's Park Station", "Museum Station", "St George Station", "Spadina Station", "Dupont Station",
        "St Clair West Station", "Cedarvale Station", "Glencairn Station", "Lawrence West Station", "Yorkdale Station",
        "Wilson Station", "Sheppard West Station", "Downsview Park Station", "Finch West Station", "York University",
        "Pioneer Village Station", "Highway 407 Station", "Vaughan Metropolitan Centre Station"],
    2: ["Kipling Station", "Islington Station", "Royal York Station", "Old Mill Station", "Jane Station",
        "Runnymede Station", "High Park Station", "Keele Station", "Dundas West Station", "Lansdowne Station",
        "Dufferin Station", "Ossington Station", "Christie Station", "Bathurst Station", "Spadina Station",
        "St George Station", "Bay Station", "Yonge Station", "Sherbourne Station", "Castle Frank Station",
        "Broadview Station", "Chester Station", "Pape Station", "Donlands Station", "Greenwood Station",
        "Coxwell Station", "Woodbine Station", "Main Street Station", "Victoria Park Station", "Warden Station",
        "Kennedy Station"],
    4: ["Sheppard-Yonge Station", "Bayview Station", "Bessarion Station", "Leslie Station", "Don Mills Station"],
    5: ["Mount Dennis Station", "Keelesdale Station", "Caledonia Station", "Fairbank Station", "Oakwood Station",
        "Cedarvale Station", "Forest Hill Station", "Chaplin Station", "Avenue Station", "Eglinton Station",
        "Mount Pleasant Station", "Leaside Station", "Laird Station", "Sunnybrook Park Station", "Don Valley Station",
        "Aga Khan Park & Museum Station", "Wynford Station", "Sloane Station", "O'Connor Station", "Pharmacy Station",
        "Hakimi Lebovic Station", "Golden Mile Station", "Birchmount Station", "Ionview Station", "Kennedy Station"],
    6: ["Humber College Station", "Westmore Station", "Martin Grove Station", "Albion Station", "Stevenson Station",
        "Mount Olive Station", "Rowntree Mills Station", "Pearldale Station", "Duncanwoods Station",
        "Milvan Rumike Station", "Emery Station", "Signet Arrow Station", "Norfinch Oakdale Station",
        "Jane and Finch Station", "Driftwood Station", "Tobermory Station", "Sentinel Station", "Finch West Station"]
}

ttc_station_list = [station for sublist in ttc_stations.values() for station in sublist]
len(ttc_stations)

5

In [15]:
def is_subway_station(stop_name: str):
    res = re.findall(r'\b(?:platform)\b', stop_name, flags=re.I)
    return any(station in stop_name for station in ttc_station_list) and bool(res)


In [16]:
# is_subway_station("Sheppard-Yonge Station - Southbound Platform")
# is_subway_station("Kipling Station - Subway Platform")

In [17]:
# def is_subway_bus_station(stop_name:str):
#     res = re.findall(r'\b(?:platform)\b', stop_name, flags=re.I)
#     return any(station in stop_name for station in ttc_station_list) and not bool(res)

In [18]:
# is_subway_bus_station("Sheppard Ave East at Burbank Dr - Bessarion Station")

In [19]:

sorted_stations = sorted(ttc_station_list, key=len, reverse=True)


def find_normalized_station(name):
    if not isinstance(name, str):
        return None
    name_lower = name.lower()
    for station in sorted_stations:
        if station.lower() in name_lower:
            return station
    return None

In [20]:
def get_line(name):
    if not is_subway_station(name):
        return None
    root_station = clean_station_name(name)
    keys = [str(k) for k, v in ttc_stations.items() if any(root_station.lower() in station.lower() for station in v)]
    return ",".join(keys) if keys else None

In [21]:
import re
'''
Assumption is that the station name is always present in the stop_name and it contains the word "station" (except for the special case of "York University").
And station names ends with Platform

Example
Kennedy Station - SouthBound Platform
Sheppard-Yonge Station - SouthBound Platform
Don Mills Station - Subway Platform - Terminal
'''
def clean_station_name(name):
    if not isinstance(name, str):
        return None
    #  this is the only exception is station name pattern present in the dataset
    if "York University" in name:
        return "York University"
    if "station" not in name.lower() or "platform" not in name.lower():
        return None


    name = re.sub(r'\b(Eastbound|Westbound|Northbound|Southbound)\b', '', name, flags=re.I)
    name = re.sub(r'\b(LRT|Bus|Platform|Stop|Terminal|Subway Platform)\b', '', name, flags=re.I)
    name = re.sub(r'\s+[^\w&]+\s+', ' ', name)

    # Normalize spacing
    name = re.sub(r'\s+', ' ', name).strip()

    return name

In [22]:
print(get_line("West Service Rd at St Albans Rd"))
get_line("Kennedy Station - Platform"), get_line("Kipling Station - Subway Platform")

None


('2,5', '2')

In [23]:
# Apply function across rows where axis=1

def generate_parent_station_id(row):
    station_name = row['parent_station']
    if pd.isna(station_name):
        return None
    # Find all stops with the same parent_station and return the min stop_id
    stations = df[df['parent_station'] == station_name]
    return stations['stop_id'].min() if not stations.empty else None

In [24]:
'''
Gets the station mode
Assumption is that if the stop name contains "at" it is a bus stop, if it contains "station" and is in the list of subway stations it is a subway or LRT station, if it contains "go" it is a GO station, otherwise - default it is a bus stop.
'''
def station_mode(stop_name):
    if " at " in stop_name:
        return 'BUS'
    line = get_line(stop_name)
    if is_subway_station(stop_name) and line is not None:
        if any(x in "1,2,4" for x in line):
            return 'SUBWAY'
        return 'LRT'
    elif "go" in stop_name.lower():
        return 'GO'
    else:
        return 'BUS'

In [25]:
station_mode("Rowntree Mills Station")
station_mode("Rowntree Mills Station Eastbound Platform")

'LRT'

In [26]:
'''
TODO :  ADD A UNIQUE STATION ID FOR EACH STATION BASED ON

FOR Subway (TBA)
LINE_LineNumber_PARENTID_STOP_CODE
------------------------------------------------
FOR TTC (UNIQUE CODE ONLY)
BUS_STOP_CODE
------------------------------------------------
FOR GO
GO_STOP_CODE
'''
def generate_station_ids(row):
    mode = row['mode']
    if mode == 'SUBWAY' or mode == 'LRT':
        return f"{mode}_{row['parent_station_id']}_{row['stop_id']}"

    return f"{mode}_{row['stop_id']}"


In [27]:

df['temp_station'] = df['stop_name'].apply(find_normalized_station)

df['parent_station'] = (df.groupby(['stop_lat', 'stop_lon'])['temp_station'].transform(
    lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None))

df['is_subway'] = df['stop_name'].apply(is_subway_station)


df['line_number'] = df['stop_name'].apply(get_line)

df['temp_station_id'] = df.groupby('parent_station')['stop_id'].transform('min')

df['parent_station_id'] = (df.groupby(['stop_lat', 'stop_lon'])['temp_station_id'].transform(
    lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None).astype('Int64'))

df['mode'] = df['stop_name'].apply(station_mode)

df.drop(columns=['temp_station'], inplace=True)
df.drop(columns=['temp_station_id'], inplace=True)

df['station_id'] = df.apply(generate_station_ids, axis=1)




In [28]:
station_mode("Kipling Station - Subway Platform"), get_line("Kipling Station - Subway Platform")

('SUBWAY', '2')

In [29]:
(df[df['parent_station'].str.contains(r'\b(?:Kipling)\b', case=False, na=False)])

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,is_subway,line_number,parent_station_id,mode,station_id
7945,14716,14716,Kipling Station,NaN,43.637209,-79.536117,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14716
7946,14717,14717,Kipling Station,NaN,43.637354,-79.535939,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14717
7947,14718,14718,Kipling Station,NaN,43.637465,-79.535810,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14718
7948,14719,14719,Kipling Station,NaN,43.637582,-79.535677,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14719
7949,14720,14720,Kipling Station,NaN,43.637717,-79.535511,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14720
7950,14721,14721,Kipling Station,NaN,43.637851,-79.535345,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14721
7951,14722,14722,Kipling Station,NaN,43.637963,-79.535215,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14722
7952,14723,14723,Kipling Station,NaN,43.638084,-79.535066,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14723
7953,14724,14724,Kipling Station,NaN,43.638195,-79.534941,NaN,NaN,NaN,Kipling Station,NaN,1,False,NaN,13785,BUS,BUS_14724
8121,13785,13785,Kipling Station - Eastbound Platform,NaN,43.637648,-79.535694,NaN,NaN,NaN,Kipling Station,NaN,1,True,2,13785,SUBWAY,SUBWAY_13785_13785


In [30]:
# print(df[df['parent_station'].str.contains(r'\b(?:kipling)\b', case=False, na=False)])
# df[df["is_subway_line"] == True]['is_subway_line'].count()

In [31]:
df.columns

Index(['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat',
       'stop_lon', 'zone_id', 'stop_url', 'location_type', 'parent_station',
       'stop_timezone', 'wheelchair_boarding', 'is_subway', 'line_number',
       'parent_station_id', 'mode', 'station_id'],
      dtype='str')

In [32]:
len(df['parent_station'].dropna().unique())

110

In [33]:
df[['parent_station','line_number']].dropna().drop_duplicates(subset=['parent_station'])['line_number'].value_counts()

line_number
1        28
2        27
5        22
6        17
4         4
1,6       3
1,5       3
1,2       2
1,4       1
1,2,4     1
2,5       1
Name: count, dtype: int64

In [34]:
df['station_id'][df['mode'] == 'Subway'].count()

np.int64(0)

In [35]:
df.columns


Index(['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat',
       'stop_lon', 'zone_id', 'stop_url', 'location_type', 'parent_station',
       'stop_timezone', 'wheelchair_boarding', 'is_subway', 'line_number',
       'parent_station_id', 'mode', 'station_id'],
      dtype='str')

In [36]:
df[df['mode'] == 'Subway']["stop_name"].head()

Series([], Name: stop_name, dtype: str)

In [37]:
(df[df["parent_station"] == "North York Centre Station"])

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,is_subway,line_number,parent_station_id,mode,station_id
6321,8984,8984,Yonge St at Empress Ave - North York Centre St...,NaN,43.769149,-79.412700,NaN,NaN,NaN,North York Centre Station,NaN,1,False,NaN,8984,BUS,BUS_8984
8058,13789,13789,North York Centre Station - Southbound Platform,NaN,43.767347,-79.412492,NaN,NaN,NaN,North York Centre Station,NaN,1,True,1,8984,SUBWAY,SUBWAY_8984_13789
8119,13790,13790,North York Centre Station - Northbound Platform,NaN,43.768547,-79.412592,NaN,NaN,NaN,North York Centre Station,NaN,1,True,1,8984,SUBWAY,SUBWAY_8984_13790


In [38]:
# df_ttc = df[df['mode'] == 'TTC'].copy()
#
# df_ttc['station'] = df_ttc['stop_name'].apply(
#     lambda x: next((s for s in ttc_stations if s in x), None)
# )

In [39]:
processed = list(map(lambda x: str(x).lower(), df['parent_station'].dropna().unique().tolist()))

In [40]:
not_found = [s for s in ttc_station_list if s.lower() not in processed]
not_found

[]

In [41]:
set(df['parent_station'].dropna().unique().tolist()) - set(ttc_station_list)

set()

In [42]:
df.shape

(9393, 17)

In [43]:
# validating if stop_id and stop_code are same
df['stop_id'].equals(df['stop_code'])
(df.stop_id == df.stop_code).all()
# The result is true, so we can safely drop stop_code
df.drop(columns=['stop_code', "stop_url", 'stop_desc', 'zone_id', 'location_type', 'stop_timezone'], inplace=True)

In [44]:
df.shape

(9393, 11)

In [45]:
df.to_csv('processed_stations.csv', index=False)
print("Saved Processed DataFrame to 'processed_stations.csv'")

Saved Processed DataFrame to 'processed_stations.csv'


In [46]:
# generated by Gemini

import folium
from folium.plugins import MarkerCluster

# Renamed 'station' to 'parent_station' to match changes above
df_stations = df.dropna(subset=['parent_station'])


# Get an arbitrary line number for color mapping if it's an interchange station
def first_line_num(lines_str):
    if isinstance(lines_str, str) and lines_str:
        return int(lines_str.split(',')[0])
    return 1


# Aggregate
station_coords = df_stations.groupby(['parent_station', 'line_number'])[['stop_lat', 'stop_lon']].mean().reset_index()

center_lat = station_coords['stop_lat'].mean()
center_lon = station_coords['stop_lon'].mean()

# Allowed Folium colors: 'red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray'
colors = {
    1: 'yellow',
    # Note: 'yellow' might not be standard Folium color, fallback used if error: we will use custom hex for lines
    2: 'green',
    4: 'purple',
    5: 'orange',
    6: 'gray'
}

marker_colors = {
    1: 'orange',  # Using supported colors
    2: 'green',
    4: 'purple',
    5: 'cadetblue',
    6: 'gray'
}

m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

marker_cluster = MarkerCluster().add_to(m)

for idx, row in station_coords.iterrows():
    f_line = first_line_num(row['line_number'])
    folium.Marker(
        location=[row['stop_lat'], row['stop_lon']],
        popup=f"{row['parent_station']} (Line {row['line_number']})",
        icon=folium.Icon(color=marker_colors.get(f_line, 'blue'), icon="info-sign"),
    ).add_to(marker_cluster)

# Draw polylines per line
# In order to draw connecting lines exactly, we iterate over the predefined lines and draw them
for line_id, st_list in ttc_stations.items():
    # Filter the subset coordinates for the specific line
    # Map station name to its coordinates
    line_coords = []
    for st_name in st_list:
        st_row = station_coords[station_coords['parent_station'] == st_name]
        if not st_row.empty:
            line_coords.append((st_row.iloc[0]['stop_lat'], st_row.iloc[0]['stop_lon']))

    # We map colors manually for Polyline
    poly_colors = {1: '#F8C300', 2: '#00923F', 4: '#B2005A', 5: '#F2602A', 6: '#A19E92'}

    if line_coords:
        folium.PolyLine(line_coords, color=poly_colors.get(line_id, 'red'), weight=4, opacity=0.8,
                        tooltip=f"Line {line_id}").add_to(m)

m.save('index.html')
# m

In [47]:
df.head()

,stop_id,stop_name,stop_lat,stop_lon,parent_station,wheelchair_boarding,is_subway,line_number,parent_station_id,mode,station_id
0,662,Danforth Rd at Kennedy Rd,43.714379,-79.260939,NaN,1,False,NaN,<NA>,BUS,BUS_662
1,929,Davenport Rd at Bedford Rd,43.674448,-79.399659,NaN,1,False,NaN,<NA>,BUS,BUS_929
2,940,Davenport Rd at Dupont St,43.675511,-79.401938,NaN,2,False,NaN,<NA>,BUS,BUS_940
3,1871,Davisville Ave at Cleveland St,43.702088,-79.378112,NaN,1,False,NaN,<NA>,BUS,BUS_1871
4,11700,Disco Rd at Attwell Dr,43.701362,-79.594843,NaN,1,False,NaN,<NA>,BUS,BUS_11700


In [48]:
df[['stop_id', 'stop_name', 'mode', 'station_id']].isna().any()

stop_id       False
stop_name     False
mode          False
station_id    False
dtype: bool

In [49]:
df[df["stop_name"].str.contains(r'\b(?:Downsview Park Station)\b', case=False, na=False)].drop(
    columns=["wheelchair_boarding", "stop_lat", "stop_lon"])



,stop_id,stop_name,parent_station,is_subway,line_number,parent_station_id,mode,station_id
7066,3106,Sheppard Ave West at Bakersfield St West Side ...,Downsview Park Station,False,NaN,3106,BUS,BUS_3106
8525,15664,Downsview Park Station - Northbound Platform,Downsview Park Station,True,1,3106,SUBWAY,SUBWAY_3106_15664
8526,15665,Downsview Park Station - Southbound Platform,Downsview Park Station,True,1,3106,SUBWAY,SUBWAY_3106_15665
8547,15631,Sheppard Ave West at Vitti St - Downsview Park...,Downsview Park Station,False,NaN,3106,BUS,BUS_15631
8912,16345,Downsview Park Trail at Downsview Park Station,Downsview Park Station,False,NaN,3106,BUS,BUS_16345


In [50]:
is_subway_station("Downsview Park Station - Northbound Platform")
is_subway_station("Sheppard Ave West at Bakersfield St West Side - Downsview Park Station")


False

In [51]:
df.head()
df[['stop_id', 'stop_name', 'mode', 'station_id']].isna().any()

stop_id       False
stop_name     False
mode          False
station_id    False
dtype: bool

In [52]:
df['parent_station_id'].dropna().drop_duplicates().count()

np.int64(110)

In [53]:
subways_stations = df[df['mode'] == 'Subway'][
    ['stop_name', 'parent_station_id', 'parent_station', 'mode']].dropna().drop_duplicates(
    subset=['parent_station_id'])
subways_stations.to_csv("subways_stations.csv", index=False)

In [54]:
df[df['parent_station_id'] == 16222]

,stop_id,stop_name,stop_lat,stop_lon,parent_station,wheelchair_boarding,is_subway,line_number,parent_station_id,mode,station_id
9016,16222,O'Connor Station Eastbound Platform,43.724820,-79.301367,O'Connor Station,1,True,5,16222,LRT,LRT_16222_16222
9017,16223,O'Connor Station Westbound Platform,43.724831,-79.302203,O'Connor Station,1,True,5,16222,LRT,LRT_16222_16223


In [55]:
df[df['stop_name'].str.contains(r'\b(?:Humber College)\b', case=False, na=False)]

,stop_id,stop_name,stop_lat,stop_lon,parent_station,wheelchair_boarding,is_subway,line_number,parent_station_id,mode,station_id
233,12196,Humber College Blvd at Humberline Dr South Side,43.731793,-79.608899,NaN,1,False,NaN,<NA>,BUS,BUS_12196
234,12064,Humberline Dr at Humber College Blvd West Side,43.731956,-79.609689,NaN,1,False,NaN,<NA>,BUS,BUS_12064
235,11049,Humberline Dr at Humber College Blvd,43.731861,-79.609562,NaN,1,False,NaN,<NA>,BUS,BUS_11049
775,7857,Humber College Blvd at Finch Ave West South Side,43.733717,-79.610132,NaN,1,False,NaN,<NA>,BUS,BUS_7857
1420,11273,Humber College Blvd at Highway 27,43.730243,-79.601847,NaN,1,False,NaN,<NA>,BUS,BUS_11273
1879,7863,Humber College Blvd at John Garland Blvd,43.733547,-79.593827,NaN,1,False,NaN,<NA>,BUS,BUS_7863
2019,11081,Humber College Blvd at Carrier Dr South Side,43.737100,-79.610515,NaN,1,False,NaN,<NA>,BUS,BUS_11081
2457,7864,Humber College Blvd at Lynmont Rd,43.731799,-79.596075,NaN,1,False,NaN,<NA>,BUS,BUS_7864
3877,7862,Humber College Blvd at John Garland Blvd West ...,43.733587,-79.594019,NaN,2,False,NaN,<NA>,BUS,BUS_7862
4088,7867,Humber College Blvd at Westmore Dr - William O...,43.731137,-79.598867,NaN,1,False,NaN,<NA>,BUS,BUS_7867


In [69]:
df[(df['stop_id'] >= 13733) & (df['stop_id'] <= 13738)]

,stop_id,stop_name,stop_lat,stop_lon,parent_station,wheelchair_boarding,is_subway,line_number,parent_station_id,mode,station_id
8147,13738,Woodbine Station - Eastbound Platform,43.686649,-79.312286,Woodbine Station,1,True,2,13737,SUBWAY,SUBWAY_13737_13738
8148,13735,Main Street Station - Eastbound Platform,43.689249,-79.300686,Main Street Station,1,True,2,13735,SUBWAY,SUBWAY_13735_13735
8149,13734,Victoria Park Station - Eastbound Platform,43.695149,-79.287885,Victoria Park Station,1,True,2,11376,SUBWAY,SUBWAY_11376_13734
8154,13733,Victoria Park Station - Westbound Platform,43.694649,-79.289485,Victoria Park Station,1,True,2,11376,SUBWAY,SUBWAY_11376_13733
8155,13736,Main Street Station - Westbound Platform,43.688949,-79.302386,Main Street Station,1,True,2,13735,SUBWAY,SUBWAY_13735_13736
8156,13737,Woodbine Station - Westbound Platform,43.686349,-79.313986,Woodbine Station,1,True,2,13737,SUBWAY,SUBWAY_13737_13737


In [70]:
df[df['stop_id']==11376]

,stop_id,stop_name,stop_lat,stop_lon,parent_station,wheelchair_boarding,is_subway,line_number,parent_station_id,mode,station_id
3023,11376,Victoria Park Ave at Victoria Park Station,43.694208,-79.289374,Victoria Park Station,1,False,NaN,11376,BUS,BUS_11376
